In [1]:
import torch
import torch.nn as nn
import torchaudio
import os
import random
import soundfile as sf
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Model

In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch:", torch.__version__)
print("Using Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.9.1+cpu
Using Device: cpu


In [3]:
class AudioDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]

        # 🔥 FAST + NO TORCHCODEC ISSUE
        waveform, sr = sf.read(file_path)

        waveform = torch.tensor(waveform).float()

        # convert stereo → mono
        if len(waveform.shape) > 1:
            waveform = waveform.mean(dim=1)

        # FIX LENGTH (3 sec)
        max_len = 16000 * 3

        if waveform.shape[0] > max_len:
            waveform = waveform[:max_len]
        else:
            pad_size = max_len - waveform.shape[0]
            waveform = torch.nn.functional.pad(waveform, (0, pad_size))

        return waveform, torch.tensor(self.labels[idx])


In [5]:
real_dir = r"D:\DEEPFAKE\datasets\audio_dataset\real"
fake_dir = r"D:\DEEPFAKE\datasets\audio_dataset\fake"
files = []
labels = []

# -------------------------
# LOAD REAL DATA
# -------------------------
for f in os.listdir(real_dir):
    files.append(os.path.join(real_dir, f))
    labels.append(0)

# -------------------------
# LOAD FAKE DATA
# -------------------------
for f in os.listdir(fake_dir):
    files.append(os.path.join(fake_dir, f))
    labels.append(1)

print("Total samples before shuffle:", len(files))

# -------------------------
# 🔥 SHUFFLE (VERY IMPORTANT)
# -------------------------
combined = list(zip(files, labels))
random.shuffle(combined)

files, labels = zip(*combined)

files = list(files)
labels = list(labels)

# -------------------------
# 🔥 LIMIT DATASET SIZE
# -------------------------
MAX_SAMPLES = 2000   # change here (1000 / 2000 / 3000)

files = files[:MAX_SAMPLES]
labels = labels[:MAX_SAMPLES]

print("Total samples used:", len(files))
print("Real samples:", labels.count(0))
print("Fake samples:", labels.count(1))

Total samples before shuffle: 21560
Total samples used: 2000
Real samples: 974
Fake samples: 1026


In [6]:
dataset = AudioDataset(files, labels)

train_loader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=0,     # ⚠️ Windows safe
    
)

In [7]:
class AudioDeepfakeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")
        self.classifier = nn.Linear(768, 2)

    def forward(self, x):
        outputs = self.wav2vec(x)
        pooled = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(pooled)


model = AudioDeepfakeModel().to(DEVICE)

C:\Users\vansh\AppData\Roaming\Python\Python313\site-packages\transformers\configuration_utils.py:335: UserWarning: Passing `gradient_checkpointing` to a config initialization is deprecated and will be removed in v5 Transformers. Using `model.gradient_checkpointing_enable()` instead, or if you are using the `Trainer` API, pass `gradient_checkpointing=True` in your `TrainingArguments`.
  warnings.warn(


In [8]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

EPOCHS = 1

In [9]:
print("\nTesting DataLoader...")
for x, y in train_loader:
    print("✅ DataLoader Working:", x.shape)
    break


Testing DataLoader...
✅ DataLoader Working: torch.Size([4, 48000])


In [10]:
# 🔥 SPLIT DATA (80-20)
split = int(0.8 * len(dataset))

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [split, len(dataset)-split])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

for epoch in range(EPOCHS):
    print(f"\n🚀 Starting Epoch {epoch+1}")

    # ---------------- TRAIN ----------------
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for i, (waveforms, labels) in enumerate(train_loader):

        waveforms = waveforms.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(waveforms)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 🔥 ACCURACY
        preds = torch.argmax(outputs, dim=1)
        batch_correct = (preds == labels).sum().item()
        batch_total = labels.size(0)

        correct += batch_correct
        total += batch_total

        batch_acc = batch_correct / batch_total

        # 🔥 PRINT EVERY FEW BATCHES
        if i % 5 == 0:
            print(f"[Epoch {epoch+1} | Batch {i}] Loss: {loss.item():.4f} | Acc: {batch_acc:.4f}")

    # 🔥 FINAL TRAIN METRICS
    train_acc = correct / total

    print(f"\n✅ Epoch {epoch+1} Summary:")
    print(f"Loss: {total_loss/len(train_loader):.4f}")
    print(f"Training Accuracy: {train_acc:.4f}")

    # ---------------- VALIDATION ----------------
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for i, (waveforms, labels) in enumerate(val_loader):

            waveforms = waveforms.to(DEVICE)
            labels = labels.to(DEVICE)

            outputs = model(waveforms)

            preds = torch.argmax(outputs, dim=1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # 🔥 OPTIONAL: batch-wise val logging
            if i % 5 == 0:
                print(f"[VAL Batch {i}] running...")

    val_acc = correct / total

    print(f"🎯 Validation Accuracy: {val_acc:.4f}")


🚀 Starting Epoch 1
[Epoch 1 | Batch 0] Loss: 0.7188 | Acc: 0.2500
[Epoch 1 | Batch 5] Loss: 0.6540 | Acc: 0.5000
[Epoch 1 | Batch 10] Loss: 0.5387 | Acc: 1.0000
[Epoch 1 | Batch 15] Loss: 0.6771 | Acc: 0.5000
[Epoch 1 | Batch 20] Loss: 0.5558 | Acc: 0.7500
[Epoch 1 | Batch 25] Loss: 0.4623 | Acc: 1.0000
[Epoch 1 | Batch 30] Loss: 0.3859 | Acc: 1.0000
[Epoch 1 | Batch 35] Loss: 0.3037 | Acc: 1.0000
[Epoch 1 | Batch 40] Loss: 0.3291 | Acc: 0.7500
[Epoch 1 | Batch 45] Loss: 0.0867 | Acc: 1.0000
[Epoch 1 | Batch 50] Loss: 0.1486 | Acc: 1.0000
[Epoch 1 | Batch 55] Loss: 0.0573 | Acc: 1.0000
[Epoch 1 | Batch 60] Loss: 0.0347 | Acc: 1.0000
[Epoch 1 | Batch 65] Loss: 0.0265 | Acc: 1.0000
[Epoch 1 | Batch 70] Loss: 0.0213 | Acc: 1.0000
[Epoch 1 | Batch 75] Loss: 0.0159 | Acc: 1.0000
[Epoch 1 | Batch 80] Loss: 0.0190 | Acc: 1.0000
[Epoch 1 | Batch 85] Loss: 0.0128 | Acc: 1.0000
[Epoch 1 | Batch 90] Loss: 0.0086 | Acc: 1.0000
[Epoch 1 | Batch 95] Loss: 0.0143 | Acc: 1.0000
[Epoch 1 | Batch 100] 

In [11]:
os.makedirs("ai_models/audio", exist_ok=True)

torch.save(model.state_dict(), "ai_models/audio/wav2vec_audio.pth")

print("\n✅ AUDIO MODEL TRAINED SUCCESSFULLY!")


✅ AUDIO MODEL TRAINED SUCCESSFULLY!
